## read the JSON file that you saved in ex02

In [21]:
import pandas as pd
import numpy as np
df = pd.read_json('../../data/auto.json', orient='records')
pd.options.display.float_format = '{:.2f}'.format


## enrich the dataframe using a sample from that dataframe

In [22]:
cars_sample = df[['CarNumber', 'Make', 'Model']].sample(
    n=200,
    replace=True,          
).reset_index(drop=True)

refund_sample = df['Refund'].sample(
    n=200,
    replace=True,
    random_state=21  
).reset_index(drop=True)

fines_sample = df['Fines'].sample(
    n=200,
    replace=True,
    random_state=21  
).reset_index(drop=True)

big_sample = pd.DataFrame({
    'CarNumber': cars_sample['CarNumber'],
    'Make': cars_sample['Make'],
    'Model': cars_sample['Model'],
    'Refund': refund_sample,
    'Fines': fines_sample
})

concat_rows = pd.concat([df, big_sample], ignore_index=True)
concat_rows.count()

CarNumber    925
Refund       925
Fines        925
Make         925
Model        914
dtype: int64

## enrich the dataframe concat_rows by a new column with the data generated

In [23]:
np.random.seed(21) 
rand_years = np.random.randint(1980, 2020, size=len(concat_rows))  
Year = pd.Series(rand_years, name="Year")

In [24]:
fines = pd.concat([concat_rows, Year], axis=1)

## enrich the dataframe with the data from another dataframe

In [25]:
#create a new dataframe with the car numbers and their owners 

In [26]:
surname = pd.read_json('../../datasets/surname.json')

In [27]:
unique_numbers = len(concat_rows['CarNumber'].unique())

In [28]:
surname.columns = surname.iloc[0]
surname = surname[1:].reset_index(drop=True)

In [29]:
sampled_surnames = surname['NAME'].sample(
    n=unique_numbers, 
    random_state=21,
    replace=True
).reset_index(drop=True)

In [30]:
car_numbers = pd.Series(concat_rows['CarNumber'].unique(), name='CarNumber')
owners = pd.concat([car_numbers, sampled_surnames ], axis=1)
owners = owners.rename(columns={'NAME': 'SURNAME'})
owners['SURNAME'] = owners['SURNAME'].str.replace(r'[\[\]"]', '', regex=True)
len(owners)

531

In [31]:
#append 5 more observations to the fines dataframe (come up with your own ideas of CarNumber, etc.)
my_data = [['K492MX777RUS', 2, 1000, 'Ford', 'Bronco', 2019],
           ['A815BC199RUS', 1, 2000, 'Subaru','Legacy',2020],
           ['M307ET161RUS',2,3000, 'Kia', 'Sportage',2019],
           ['X554OH750RUS',1,4000,'Ford','Explorer',2020],
           ['P921KA123RUS',2,5000,'Toyota','Tacoma',2019]]
my_df = pd.DataFrame(my_data, columns=fines.columns)
fines = pd.concat([fines, my_df], ignore_index=True)
fines.count()

CarNumber    930
Refund       930
Fines        930
Make         930
Model        919
Year         930
dtype: int64

In [32]:
len(fines)

930

In [33]:
#delete the last 20 observations from owners
owners = owners.iloc[:-20]
#add 3 new observations
my_new_data = [['Y662MT777RUS','DU BOIS'],
           ['M084XK61RUS', 'KITSURAGI'],
           ['H958KH36RUS','VICQUEMARE']]
my_new_df = pd.DataFrame(my_new_data, columns=owners.columns)
owners = pd.concat([owners, my_new_df], ignore_index=True)

In [34]:
#join both dataframes
#the new dataframe should have only the car numbers that exist in both dataframes
pd.merge(fines, owners, on='CarNumber', how='inner')

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989,RICHARDSON
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995,ROSS
2,7184TT36RUS,1,2100.00,Ford,Focus,1984,MORGAN
3,X582HE161RUS,2,2000.00,Ford,Focus,2015,BAILEY
4,92918M178RUS,1,5700.00,Ford,Focus,2014,LOPEZ
...,...,...,...,...,...,...,...
896,708487163RUS,1,2000.00,Ford,Focus,1981,HARRIS
897,8603T8154RUS,2,400.00,Ford,Focus,1992,MITCHELL
898,Y255E877RUS,1,12800.00,Ford,Focus,2007,WATSON
899,7064C8197RUS,2,800.00,Volkswagen,Passat,2005,DAVIS


In [35]:
## the new dataframe should have all the car numbers that exist in both dataframes
pd.merge(fines, owners, on='CarNumber', how='outer')

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,704687163RUS,2.00,1400.00,Ford,Focus,2004.00,ADAMS
1,704687163RUS,2.00,2600.00,Ford,Focus,1997.00,ADAMS
2,704787163RUS,2.00,2800.00,Ford,Focus,1992.00,MORGAN
3,704787163RUS,2.00,8594.59,Ford,Focus,1988.00,MORGAN
4,704987163RUS,2.00,8594.59,Ford,Focus,1985.00,MITCHELL
...,...,...,...,...,...,...,...
928,Y969O8197RUS,2.00,18000.00,Ford,Focus,1984.00,LOPEZ
929,Y973O8197RUS,2.00,8594.59,Ford,Focus,2005.00,YOUNG
930,Y973O8197RUS,1.00,34800.00,Ford,Focus,2003.00,YOUNG
931,Y973O8197RUS,1.00,69600.00,Ford,Focus,2017.00,YOUNG


In [36]:
## the new dataframe should have only the car numbers from the fines dataframe
pd.merge(fines, owners, on='CarNumber', how='left')

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989,RICHARDSON
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995,ROSS
2,7184TT36RUS,1,2100.00,Ford,Focus,1984,MORGAN
3,X582HE161RUS,2,2000.00,Ford,Focus,2015,BAILEY
4,92918M178RUS,1,5700.00,Ford,Focus,2014,LOPEZ
...,...,...,...,...,...,...,...
925,K492MX777RUS,2,1000.00,Ford,Bronco,2019,NaN
926,A815BC199RUS,1,2000.00,Subaru,Legacy,2020,NaN
927,M307ET161RUS,2,3000.00,Kia,Sportage,2019,NaN
928,X554OH750RUS,1,4000.00,Ford,Explorer,2020,NaN


In [37]:
## the new dataframe should have only the car numbers from the owners dataframe
pd.merge(fines, owners, on='CarNumber', how='right')

,CarNumber,Refund,Fines,Make,Model,Year,SURNAME
0,Y163O8161RUS,2.00,3200.00,Ford,Focus,1989.00,RICHARDSON
1,Y163O8161RUS,2.00,1600.00,Ford,Focus,1980.00,RICHARDSON
2,Y163O8161RUS,1.00,1600.00,Ford,Focus,1993.00,RICHARDSON
3,Y163O8161RUS,1.00,1500.00,Ford,Focus,2009.00,RICHARDSON
4,E432XX77RUS,1.00,6500.00,Toyota,Camry,1995.00,ROSS
...,...,...,...,...,...,...,...
899,O50197197RUS,2.00,7800.00,Ford,Focus,1992.00,WRIGHT
900,7608EE777RUS,1.00,4000.00,Skoda,Octavia,2000.00,HILL
901,Y662MT777RUS,NaN,NaN,NaN,NaN,NaN,DU BOIS
902,M084XK61RUS,NaN,NaN,NaN,NaN,NaN,KITSURAGI


## create a pivot table from the fines dataframe

In [38]:
pivot = fines.pivot_table(values=['Fines'], index=['Make', 'Model'], columns=['Year'])
pivot

Fines                                             \
Year                    1980     1981    1982    1983     1984     1985   
Make       Model                                                          
Ford       Bronco        NaN      NaN     NaN     NaN      NaN      NaN   
           Explorer      NaN      NaN     NaN     NaN      NaN      NaN   
           Focus     4416.07 18618.53 7019.19 4853.85  7499.64  7142.08   
           Mondeo        NaN      NaN     NaN     NaN      NaN      NaN   
Kia        Sportage      NaN      NaN     NaN     NaN      NaN      NaN   
Skoda      Octavia   1900.00      NaN 3450.00 3864.86      NaN  3431.53   
Subaru     Legacy        NaN      NaN     NaN     NaN      NaN      NaN   
Toyota     Camry    12000.00  8594.59     NaN 7200.00      NaN      NaN   
           Corolla       NaN  7600.00 2000.00     NaN      NaN 39600.00   
           Tacoma        NaN      NaN     NaN     NaN      NaN      NaN   
Volkswagen Golf     30900.00      NaN     NaN 8594.59   300.00 24000.00   
           Jetta         NaN      NaN     NaN     NaN      NaN      NaN   
           Passat        NaN  1600.00     NaN 3200.00 10000.00  5000.00   
           Touareg       NaN      NaN     NaN     NaN      NaN  5800.00   

                                                       ...                   \
Year                    1986    1987    1988     1989  ...    2011     2012   
Make       Model                                       ...                    
Ford       Bronco        NaN     NaN     NaN      NaN  ...     NaN      NaN   
           Explorer      NaN     NaN     NaN      NaN  ...     NaN      NaN   
           Focus     7499.61 7811.76 5166.07  5985.71  ... 5099.36  6431.58   
           Mondeo        NaN     NaN     NaN  8600.00  ...     NaN 34400.00   
Kia        Sportage      NaN     NaN     NaN      NaN  ...     NaN      NaN   
Skoda      Octavia    600.00 5200.00 3000.00 45700.00  ...  500.00   500.00   
Subaru     Legacy        NaN     NaN     NaN      NaN  ...     NaN      NaN   
Toyota     Camry         NaN     NaN     NaN 22400.00  ...  300.00  8594.59   
           Corolla       NaN 8000.00     NaN  4000.00  ... 8594.59      NaN   
           Tacoma        NaN     NaN     NaN      NaN  ...     NaN      NaN   
Volkswagen Golf          NaN 9300.00     NaN  5800.00  ... 1650.00      NaN   
           Jetta         NaN     NaN     NaN      NaN  ...     NaN      NaN   
           Passat   15000.00 7633.33     NaN      NaN  ...     NaN      NaN   
           Touareg       NaN     NaN     NaN      NaN  ...     NaN      NaN   

                                                                          \
Year                   2013     2014     2015     2016     2017     2018   
Make       Model                                                           
Ford       Bronco       NaN      NaN      NaN      NaN      NaN      NaN   
           Explorer     NaN      NaN      NaN      NaN      NaN      NaN   
           Focus    5528.38  6152.35 10184.73  4537.16 11383.33 18946.31   
           Mondeo       NaN      NaN      NaN 46200.00      NaN      NaN   
Kia        Sportage     NaN      NaN      NaN      NaN      NaN      NaN   
Skoda      Octavia  3448.65   300.00 23197.29  6650.00      NaN 52066.67   
Subaru     Legacy       NaN      NaN      NaN      NaN      NaN      NaN   
Toyota     Camry     300.00      NaN      NaN      NaN      NaN  7000.00   
           Corolla      NaN 43600.00  8594.59      NaN  9600.00      NaN   
           Tacoma       NaN      NaN      NaN      NaN      NaN      NaN   
Volkswagen Golf         NaN      NaN  2300.00      NaN      NaN      NaN   
           Jetta        NaN      NaN      NaN      NaN      NaN      NaN   
           Passat       NaN      NaN   600.00  1050.00      NaN      NaN   
           Touareg      NaN  1300.00   500.00      NaN      NaN      NaN   

                                     
Year                   2019    2020  
Make       Model                  

In [39]:
fines.to_csv('../../data/fines.csv', index=False)
owners.to_csv('../../data/owners.csv', index=False)